# Final Model Training and Inference Validation

## Objective

Train the provisionally selected LightGBM configuration on the complete labeled dataset and prepare it for reproducible inference.

## Modeling Decision

LightGBM was selected based on its superior validation and cross-validation ROC-AUC compared with Logistic Regression.

The final model will use 7 leaves per tree and 1,726 boosting iterations, based on the preceding experiments.

All 200,000 labeled observations will be used for training. The selected number of boosting iterations is retained as a fixed configuration; it is not guaranteed to be optimal for the larger training set.

No additional hyperparameter tuning will be performed at this stage.

In [1]:
from pathlib import Path

import lightgbm as lgb
import pandas as pd

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"

train_full = pd.read_csv(train_path)

feature_columns = [f"var_{i}" for i in range(200)]

X_full = train_full[feature_columns]
y_full = train_full["target"]

final_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1726,
    learning_rate=0.05,
    num_leaves=7,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

final_model.fit(X_full, y_full)

print(f"Training samples: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Trained trees: {final_model.n_estimators_}")
print(f"Classes: {final_model.classes_}")

Training samples: 200000
Number of features: 200
Trained trees: 1726
Classes: [0 1]


In [2]:
import joblib
import numpy as np

model_path = project_root / "models" / "lightgbm_final.joblib"

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, model_path)

loaded_model = joblib.load(model_path)

sample_features = X_full.iloc[:5]

original_proba = final_model.predict_proba(sample_features)[:, 1]
loaded_proba = loaded_model.predict_proba(sample_features)[:, 1]

predictions_match = np.allclose(
    original_proba,
    loaded_proba,
    rtol=0,
    atol=1e-12,
)

print(f"Model saved to: {model_path}")
print(f"Model size: {model_path.stat().st_size / (1024 ** 2):.2f} MB")
print(f"Predictions match: {predictions_match}")
print(f"Loaded model classes: {loaded_model.classes_}")

Model saved to: /Users/ebrietas/Santander-Customer-Transaction-Prediction/models/lightgbm_final.joblib
Model size: 1.53 MB
Predictions match: True
Loaded model classes: [0 1]


### Model Serialization and Verification

**Objective:** Verify that the trained LightGBM model can be persisted and reloaded without altering its predictions.

**Methodology:** The final model was serialized using Joblib and subsequently reloaded. Positive-class probabilities were generated for five training samples using both the original and reloaded models. Numerical consistency was checked using `numpy.allclose()` with an absolute tolerance of `1e-12` and zero relative tolerance.

**Findings:**
- The model was successfully saved and reloaded.
- The serialized model size was approximately 1.53 MB.
- Predictions matched within the specified numerical tolerance.
- The reloaded model retained the expected class order: `[0, 1]`.

**Interpretation:** The test confirms prediction consistency following serialization and reloading for the examined samples. It does not establish predictive accuracy or generalization performance on unseen data.

In [5]:
test_full = pd.read_csv(test_path)

X_test = test_full[feature_columns]

assert X_test.columns.tolist() == loaded_model.feature_name_

test_proba = loaded_model.predict_proba(X_test)[:, 1]

print("Test shape:", X_test.shape)
print("Predictions shape:", test_proba.shape)
print("All predictions finite:", np.isfinite(test_proba).all())
print("Probabilities within [0, 1]:", ((test_proba >= 0) & (test_proba <= 1)).all())
print("First five probabilities:", test_proba[:5])

Test shape: (200000, 200)
Predictions shape: (200000,)
All predictions finite: True
Probabilities within [0, 1]: True
First five probabilities: [0.07416085 0.25379128 0.22920235 0.16789351 0.05336824]


### Final Model Inference Validation

**Objective**

Verify that the serialized LightGBM model can generate valid predictions on the complete unlabeled test dataset.

**Methodology**

The reloaded model was used to predict positive-class probabilities for all 200,000 test observations. Feature names and ordering were verified against the trained model. Prediction shape, numerical finiteness, and probability bounds were also checked.

**Findings**

- All 200,000 test observations were processed successfully.
- The input contained the expected 200 numerical features.
- The output contained one predicted probability per observation.
- All predictions were finite and within the interval [0, 1].
- Feature ordering matched the model's training schema.

**Interpretation**

The results confirm that the serialized model can perform batch inference on the supplied test dataset without structural or numerical validation failures.

**Limitations**

Because the test dataset does not contain target labels, predictive performance cannot be evaluated on these observations. Successful inference does not establish probability calibration, deployment reliability, or performance on future customer populations.

**Engineering Decision**

Retain the serialized LightGBM model as the current final artifact. No competition submission is required. Remaining work will focus on repository hygiene, dependency completeness, documentation, and reproducibility.

In [4]:
submission_df = pd.DataFrame({
    "ID_code": test_full["ID_code"],
    "target": test_proba,
})

submission_path = project_root / "data" / "submissions" / "lightgbm_submission.csv"

submission_path.parent.mkdir(parents=True, exist_ok=True)

submission_df.to_csv(submission_path, index=False)

print("Submission shape:", submission_df.shape)
print("Columns:", submission_df.columns.tolist())
print("Missing values:", submission_df.isna().sum().sum())
print("File exists:", submission_path.is_file())

submission_df.head()

Submission shape: (200000, 2)
Columns: ['ID_code', 'target']
Missing values: 0
File exists: True


,ID_code,target
0,test_0,0.074161
1,test_1,0.253791
2,test_2,0.229202
3,test_3,0.167894
4,test_4,0.053368


### External Evaluation: Kaggle Test Results

**Objective**

Evaluate the finalized LightGBM model against the competition's hidden test labels and compare its performance with the internal validation estimates.

**Methodology**

The LightGBM classifier was retrained on all 200,000 labeled training observations using the previously selected configuration of 7 leaves per tree and 1,726 boosting iterations.

Positive-class probabilities were generated for the 200,000 unlabeled test observations and submitted to Kaggle for external evaluation.

No additional model tuning was performed using leaderboard feedback.

**Results**

| Evaluation Method | ROC-AUC |
|---|---:|
| Holdout Validation | 0.89450 |
| Three-Fold Cross-Validation (Mean) | 0.89310 |
| Kaggle Public Leaderboard | 0.89522 |
| Kaggle Private Leaderboard | 0.89217 |

**Findings and Interpretation**

The final LightGBM model achieved a Private Leaderboard ROC-AUC of 0.89217, closely matching the performance observed during internal validation and cross-validation.

The relatively small differences between internal and external evaluation results provide additional evidence that the model generalizes beyond the labeled training dataset.

**Final Modeling Decision**

LightGBM is retained as the final selected model. Further hyperparameter tuning is not pursued, avoiding additional adaptation to leaderboard feedback.

**Limitations**

The Kaggle scores represent performance on hidden competition evaluation subsets rather than a direct measurement across all future customer populations. Potential distribution shifts and probability calibration remain outside the scope of the current evaluation.